<h2>Optimalisatie</h2>


In [1]:
!pip install openpyxl


[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import glob

# Geef het pad naar de map met CSV-bestanden op
category = "SENH"
folder_path = './'+category+'/'
export_name = category + "_Ranking"

# Lijst met de bestandsnamen van de CSV-bestanden in de map
file_list = glob.glob(folder_path + '*.csv')

# ponytail: sentinel groter dan elke realistische wedstrijdtijd, zodat DNF/DNS/DSQ nooit als snelste telt
DNF_TIME = 9999.0

# Functie om tijd in het formaat 'mm:ss.milliseconds' naar seconden om te zetten
def time_to_seconds(time_str):
    try:
        return sum(float(x) * 60**i for i, x in enumerate(reversed(time_str.split(':'))))
    except (ValueError, AttributeError):
        return None  # DNF/DNS/DSQ of ontbrekende tijd

# Functie om punten te berekenen op basis van de ranking
def calculate_points(rank, time):
    if pd.isna(time) or not (1 <= rank <= 5):
        return 0
    return 6 - rank

# Lees de CSV-bestanden in en voeg ze samen
dfs = [pd.read_csv(file, usecols=lambda col: col not in ['Bib', 'Country', 'Score', 'Diff', 'Total'], na_values=['']).fillna(0).copy() for file in file_list]
resultsCombined = pd.concat(dfs, ignore_index=True)

# Zet de 'Time'-kolom om naar seconden; DNF/DNS/DSQ wordt voorlopig NaN
resultsCombined['Time'] = resultsCombined['Time'].apply(lambda x: time_to_seconds(x))

# Voeg een nieuwe kolom "Points" toe (DNF/DNS/DSQ = 0 punten, ongeacht de ruwe Rank)
resultsCombined['Points'] = [calculate_points(r, t) for r, t in zip(resultsCombined['Rank'], resultsCombined['Time'])]

# Vervang NaN-tijd (DNF/DNS/DSQ) nu door de sentinel, zodat ze nooit als snelste tijd tellen
resultsCombined['Time'] = resultsCombined['Time'].fillna(DNF_TIME)

# Maak een lege resultsOverview-tabel
resultsOverview = pd.DataFrame(columns=['Name', 'Event', 'Rank1', 'Time1', 'Points1', 'Rank2', 'Time2', 'Points2'])

# Groepeer resultaten op naam en evenement
grouped = resultsCombined.groupby(['Name', 'Event'])

rows_to_add = []
for group_name, group_data in grouped:
    name, event = group_name
    ranks = group_data['Rank'].values
    times = group_data['Time'].values
    points = group_data['Points'].values

    row_data = {
        'Name': name,
        'Event': event,
        'Rank1': ranks[0],
        'Time1': times[0],
        'Points1': points[0],
    }

    if len(ranks) > 1:
        row_data['Rank2'] = ranks[1]
        row_data['Time2'] = times[1]
        row_data['Points2'] = points[1]

    rows_to_add.append(row_data)

resultsOverview = pd.DataFrame(rows_to_add)

# Vul NaN-waarden in met 0
resultsOverview.fillna(0, inplace=True)

# Voeg de kolommen "TotalPoints" en "TotalTime" toe
resultsOverview['TotalPoints'] = resultsOverview['Points1'] + resultsOverview['Points2']
resultsOverview['TotalTime'] = resultsOverview['Time1'] + resultsOverview['Time2']

# Sorteer de tabel op basis van TotalPoints en TotalTime
resultsOverview.sort_values(by=['TotalPoints', 'TotalTime'], ascending=[False, True], inplace=True)

# Opnieuw indexeren van de gesorteerde tabel
resultsOverview.reset_index(drop=True, inplace=True)

# Voeg de "Rank" kolom toe op basis van de gesorteerde volgorde
resultsOverview['Rank'] = resultsOverview.index + 1

# Voeg de kolommen 'Final' en 'Lane' toe aan resultsOverview
resultsOverview['Final'] = 0
resultsOverview['Lane'] = 0

# Bepaal het aantal finales (elke finale heeft 6 lanes)
aantal_finales = len(resultsOverview) // 6 + 1

# Loop door de finales en wijs atleten toe aan de finales en lanes op basis van hun ranking
for finale in range(1, aantal_finales + 1):
    start_index = (finale - 1) * 6
    end_index = min(finale * 6, len(resultsOverview))
    
    # Sorteer atleten binnen elke finale op basis van ranking
    ranks_in_finale = resultsOverview['Rank'].iloc[start_index:end_index]
    
    # Bepaal de lanes en de finale op basis van de ranking binnen de finale
    lanes = [3, 4, 2, 5, 1, 6][:end_index - start_index]  # Zorg ervoor dat lanes de juiste lengte heeft
    final_values = [finale] * (end_index - start_index)
    
    # Wijs de juiste lane en finale toe aan atleten binnen de finale
    resultsOverview.loc[start_index:end_index-1, 'Lane'] = lanes  # -1 om ervoor te zorgen dat lanes en final_values gelijke lengtes hebben
    resultsOverview.loc[start_index:end_index-1, 'Final'] = final_values

# Toon de gesorteerde resultsOverview-tabel met de nieuwe "Rank" kolom
resultsOverview.to_excel(export_name + ".xlsx", index=False, engine='openpyxl')
